# LFM2.5-350M Finetuning — Middle School Tutor
Run on Kaggle with a T4 GPU. Upload `training_data_clean.jsonl` before running.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q "trl>=0.13.0" sentencepiece

In [ ]:
# Cell 2 — Imports
import json
import trl
print("trl version:", trl.__version__)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

In [ ]:
# Cell 3 — Constants
BASE_MODEL  = "LiquidAI/LFM2.5-350M"
DATA_PATH   = "/kaggle/input/tutor-dataset/training_data_clean.jsonl"
OUTPUT_DIR  = "/kaggle/working/lfm-tutor"
MAX_SEQ_LEN = 1024

In [ ]:
# Cell 4 — Load raw records from JSONL
def load_records(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

records = load_records(DATA_PATH)
print(f"Loaded {len(records)} samples")

In [ ]:
# Cell 5 — Load tokenizer and base model
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.model_max_length = MAX_SEQ_LEN

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")
model.config.use_cache = False

print("Model device:", next(model.parameters()).device)

In [ ]:
# Cell 6 — Inspect model layer names to find LoRA target modules
# LFM2.5 may name attention layers differently from standard transformers.
# This prints all named modules so we can identify the right ones.
for name, module in model.named_modules():
    print(name)

In [ ]:
# Cell 7 — Format each sample using LFM2.5's chat template
def format_sample(record):
    messages = [
        {"role": "user",      "content": record["question"]},
        {"role": "assistant", "content": record["answer"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

print(format_sample(records[0]))

In [ ]:
# Cell 8 — Build HuggingFace Dataset
texts = [format_sample(r) for r in records]
dataset = Dataset.from_dict({"text": texts})
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(f"Train: {len(dataset['train'])}  |  Eval: {len(dataset['test'])}")

In [ ]:
# Cell 9 — LoRA config
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 10 — Training arguments
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    warmup_steps=50,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    report_to="none",
    dataset_text_field="text",
)

In [ ]:
# Cell 11 — Train
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# Cell 12 — Merge LoRA adapters and save
model = model.merge_and_unload()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}/")

In [ ]:
# Cell 13 — Quick inference test
model.eval()

test_q = "Why do plants need sunlight?"
messages = [{"role": "user", "content": test_q}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=300, do_sample=False)
response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(response)

In [ ]:
# Cell 14 — Convert to GGUF and quantize to Q4_K_M
import os
os.chdir("/kaggle/working")

!git clone https://github.com/ggerganov/llama.cpp --depth 1
!pip install -q -r llama.cpp/requirements.txt

!python llama.cpp/convert_hf_to_gguf.py /kaggle/working/lfm-tutor/ \
    --outtype f16 \
    --outfile /kaggle/working/lfm-tutor-f16.gguf

!cmake llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DLLAMA_BUILD_TESTS=OFF
!cmake --build llama.cpp/build --target llama-quantize -j 4

!./llama.cpp/build/bin/llama-quantize \
    /kaggle/working/lfm-tutor-f16.gguf \
    /kaggle/working/lfm-tutor-q4.gguf \
    Q4_K_M

In [ ]:
# Cell 15 — Verify GGUF file size
import os

gguf_path = "/kaggle/working/lfm-tutor-q4.gguf"
size_mb = os.path.getsize(gguf_path) / 1e6
print(f"GGUF file size: {size_mb:.0f} MB")
print("Download this file and place it at: offline_chatbot/models/lfm-tutor-q4.gguf")